# SAC（Soft Actor-Critic）学习笔记

SAC 是"**连续动作 + off-policy + 最大熵**"的 Actor-Critic，也是本学习路线里连续控制主线的收官算法。它把前面学的三块知识焊在了一起：

- 从「策略梯度学习笔记」继承了 **Actor-Critic 架构**（同时学策略和价值）和"直接学策略"的路线；
- 从「DQN 学习笔记」继承了 **经验回放池 + 目标网络**（off-policy 的两大件）；
- 从 A2C/PPO 的**熵正则**出发，把"损失里的小配菜"升级成"写进贝尔曼方程的整个目标"——这就是 "Soft" 的来历。

> 配套代码：本文件夹的 `SAC_train.py`（训练）、`SAC_test.py`（动画测试）。
> 环境：Pendulum-v1（状态 3 维 (cosθ, sinθ, 角速度)，动作 1 维连续扭矩 ∈ [-2,2]，每回合固定 200 步，回报范围约 [-1200 乱转, -150 优秀]）。
> 建议阅读顺序：先按本文二~四章把四个核心公式看懂，再对照第六章的"代码 ↔ 公式"表逐行读 `SAC_train.py`。


## 目录

| 章节 | 内容 | 新旧程度 |
|---|---|---|
| 〇 | 已学知识回顾：SAC 直接复用的零件 | 旧（简述） |
| 一 | 三个动机：连续动作 / 样本效率 / 探索 | 混合 |
| 二 | 最大熵框架：熵 → 软 Q → 软贝尔曼方程 | **全新（核心）** |
| 三 | 策略网络：高斯策略 / 重参数化 / tanh 修正 | **全新（核心）** |
| 四 | 三个损失函数：Q / π / α 的推导与直觉 | **全新（核心）** |
| 五 | 稳定化设计：双 Q、软更新、数值技巧 | 新 + 呼应 DDQN |
| 六 | 代码 ↔ 公式对照（`update()` 四步） | 实现对照 |
| 七 | 实测观察：训练日志怎么读 | 实践 |
| 八 | 与已学算法的总对比 + 一句话总结 | 综合 |


## 〇、已学知识回顾：SAC 里"直接复用"的零件（简述）

| 已学零件 | 出处 | 在 SAC 里的角色 |
|---|---|---|
| 贝尔曼自举（TD 目标） | RL 基础 / DQN | Q 目标骨架 $r+\gamma Q'$，SAC 仅在中间"减一项熵" |
| 经验回放池 | DQN 笔记 | 原样复用（池子更大、采样更频繁，代码改用 numpy 环形缓冲提速） |
| 目标网络 | DQN 笔记原理二 | 复用；更新方式从"每 10 回合硬拷贝"改为"每步软更新"（5.2 节） |
| Actor + Critic 双网络 | A2C / PPO | 复用"策略 + 价值"同时学；策略输出从 `Categorical` 换成 `Normal` |
| 熵正则 | A2C / PPO | **升级**：从损失里的附加项 → 写进目标函数本身（第二章） |
| 策略梯度 + log 技巧 | REINFORCE | 换形式：SAC 不走 log 技巧路线，改用重参数化直传（3.3 节） |
| 重参数化技巧 | Noisy Net 笔记 | 同款思想：把随机性交给外部噪声 $\xi$（3.3 节会明显呼应） |
| Double Q 思想 | DDQN 笔记 | **升级**：从"选评解耦"到"双 Q 取 min"（5.1 节） |
| "连续动作怎么办"的伏笔 | 策略梯度笔记 1.1.4 | 当时表格里预留的"高斯策略"，第三章正式落实 |

> **一句话**：SAC 真正全新的东西只有两块——① 把熵写进贝尔曼方程（第二章）；② 连续高斯策略的梯度怎么传（第三、四章）。其余都是老零件的重新焊接。


## 一、三个动机：SAC 为什么长这样

### 1.1 动机一：连续动作——argmax 换成"采样"

复习「策略梯度学习笔记 1.1.2」：DQN 系"先学 Q 值再 $\arg\max_a$"，动作一旦连续（扭矩可取区间内任意实数），候选动作有无穷多个，**无法枚举**；A2C/PPO 里用的 `Categorical` 分布也只是给离散动作打分。

连续动作的解法：**把策略参数化成连续的分布**——

- 网络输出高斯分布的两个参数：均值 $\mu$ 和标准差 $\sigma$；
- 决策 = 从 $\mathcal{N}(\mu,\sigma^2)$ 里**采样**一个实数；
- Pendulum 例子：动作是 1 维扭矩，$\mu$ 表示"想给多大的力矩"，$\sigma$ 表示"有多不确定"。

| | A2C / PPO（本路线实现的版本） | SAC |
|---|---|---|
| 策略输出 | `Categorical(logits)`（离散概率） | `Normal(μ, σ)` + tanh 压缩（连续） |
| 决策方式 | 按概率选一个动作 | 从分布采样一个实数值 |
| 探索来源 | 熵正则 + 概率采样 | 分布自身的 $\sigma$ + 最大熵目标 |

### 1.2 动机二：样本效率——off-policy 的回放池回来了

- PPO 是 **on-policy**：数据必须"新鲜"，同一批数据最多复用几个 epoch 就丢；
- SAC 是 **off-policy**：数据全部进池子、反复抽。为什么可以？——**贝尔曼方程给的权利**：

  目标 $y$ 里的 $Q'$ 只是"当前策略的软 Q 值估计"，回归它并不要求数据由当前策略产生（这正是 Q 学习能 off-policy 的根本原因；DQN 笔记 1.4 节的优化问题里，期望本来就是对回放池分布取的）。

- 代价：off-policy + 自举 = 移动靶更凶（DQN 笔记原理二的问题在 SAC 里同样存在）→ 用"双 Q + 软更新"还债（第五章）；
- 代码体现：SAC **每走一步就 update 一次**（不等回合结束、不攒批），池子常年存十万条数据。

### 1.3 动机三：探索——从"外挂噪声"到"目标里长出来的随机性"

| 阶段 | 方法 | 局限 |
|---|---|---|
| DQN | ε-贪婪 | 按固定概率乱走，不看状态；要手调衰减曲线 |
| A2C / PPO | 熵正则 $-\beta H$ | 只是损失里的"配菜"，**最优策略仍是确定性** |
| **SAC** | 最大熵目标 | 熵本身就是目标的一部分，**最优策略天生带随机** |

最大熵框架下，最优策略不是"选 Q 值最大的那个动作"，而是"**按 Q 值高低分配概率**"——Q 大的动作概率大，但不为 100%。好处：探索更系统（不是无差别乱走）；对噪声/模型误差更鲁棒；同一状态可以保留多种"都还行"的行为。


## 二、最大熵框架（全新，核心）

### 2.1 熵：一个分布的"随机程度"

离散分布：$H(\pi(\cdot|s)) = -\sum_a \pi(a|s)\log\pi(a|s)$

连续分布：$H(\pi(\cdot|s)) = -\int \pi(a|s)\log\pi(a|s)\,da \;=\; \mathbb{E}_{a\sim\pi}\big[-\log\pi(a|s)\big]$

三条性质（对照记忆）：

- **确定性**策略（某动作概率 1）：$H=0$（一点都不随机）；
- **均匀**分布："最随机"，熵最大；
- 代码里所有形如 $\log\pi(a|s)$ 的项，都是"单步熵"的采样估计——**SAC 的每个损失里都有它**，这就是和前面算法的最大区别。

直觉：从分布里采样时"平均有多意外"。低概率的事件越常发生，熵越大。

### 2.2 最大熵目标：把熵加进回报

标准 RL 目标（REINFORCE / A2C / PPO 都在最大化它）：

$$J(\pi)=\sum_t \mathbb{E}\big[r(s_t,a_t)\big]$$

最大熵 RL 目标（SAC）：

$$J(\pi)=\sum_t \mathbb{E}\Big[\underbrace{r(s_t,a_t)+\alpha\,H(\pi(\cdot|s_t))}_{\text{回报 + 熵奖励}}\Big]
\;=\;\sum_t \mathbb{E}\Big[r(s_t,a_t)-\alpha\log\pi(a_t|s_t)\Big]$$

（右边是"用单个采样动作代替熵积分"后的形式，也是代码里实际用的形式。）

- $\alpha$ 叫**温度**：$\alpha\to 0$ 退化为标准 RL（只要回报）；$\alpha$ 越大越"看重随机"。
- 与 A2C/PPO 熵正则的**本质区别**：
  - A2C/PPO：`loss = -log π·A - βH`，熵是**辅助项**——影响训练过程（防坍缩），但目标函数的最优解仍是确定性策略；标准 PPO/A2C 里的熵正则，通常只“外挂”在 actor 的损失函数上，用来更新策略网络参数；它不进入 critic 的 TD target / Bellman 目标。
  - SAC：熵进入了**目标函数本身**，因此紧接着 V、Q 的定义都要跟着改（2.3 节），最优策略天然带随机性。
  - Actor：策略网络，决定“做什么”。
  - Critic：价值网络，评价“这个状态/动作有多好”。
- 为什么合理：可以把 $-\alpha\log\pi$ 读成"策略因为保持随机而获得的**每步额外奖励**"——探索不再是对回报的妥协，而是回报的一部分。

### 2.3 软价值函数与软贝尔曼方程

在最大熵框架里，价值函数的定义也要"软"化：

$$V(s)=\mathbb{E}_{a\sim\pi}\big[Q(s,a)-\alpha\log\pi(a|s)\big] \qquad \text{（软 V：“未来回报”减去“熵惩罚”）}$$

$$Q(s,a)=r(s,a)+\gamma\,\mathbb{E}_{s'}\big[V(s')\big] \qquad \text{（Q 的定义形状不变）}$$

把 $V$ 代入 $Q$，得到 **软贝尔曼方程**（SAC 的基石）：

$$Q(s,a)=r+\gamma\,\mathbb{E}_{s',a'\sim\pi}\big[Q(s',a')-\alpha\log\pi(a'|s')\big]$$

对比标准贝尔曼方程 $Q = r+\gamma\mathbb{E}[Q']$，唯一变化是：**括号里多减了一个 $\alpha\log\pi(a'|s')$**。

为什么叫"软"：标准框架里 $V$ 相当于"对 Q 做硬选择"（挑最大）；软框架里 $V$ 对 Q 的利用是"软"的——可以证明（对分布做约束优化，拉格朗日乘子一步）最优分布是 Boltzmann 形式：

$$\pi(a|s)\;\propto\;\exp\!\big(Q(s,a)/\alpha\big)$$

即"Q 越大的动作概率越高，但不为 0"。白话：不是"只盯着最好的动作"，而是"好的多做、差的少做，且始终保留随机性"。

### 2.4 软贝尔曼方程 → 代码里的目标 $y$

把三种算法 Q 目标的形状并排看：

| 算法 | 目标 $y$ |
|---|---|
| DQN（离散） | $y=r+\gamma\max_{a'}Q'(s',a')$ |
| 连续确定性 AC（如 DDPG） | $y=r+\gamma\,Q'\big(s',\mu'(s')\big)$ |
| SAC | $y=r+\gamma\Big[\min(Q_1',Q_2')\big(s',a'\big)-\alpha\log\pi(a'\mid s')\Big],\quad a'\sim\pi$ |

三个变化逐一拆解：

1. $\max$ → $\min$：双 Q 保守化（5.1 节）；
2. 多出的 $-\alpha\log\pi(a'|s')$：**软贝尔曼方程的熵项**（本章）；
3. $a'$ 用**当前策略采样**得到：因为 $V$ 的定义里有"对当前策略求期望"，用一次采样来近似这个期望（随机梯度估计）。

> 所以 `SAC_train.py` 里 `backup = rewards + GAMMA * (1-dones) * q_next`（其中 `q_next = min(Q1',Q2') - alpha*log_pi`）这一行，就是把软贝尔曼方程原样搬进了代码。


## 三、策略网络（Actor）：高斯策略的四个技术细节

### 3.1 网络输出什么：μ 和 log σ

结构上与 A2C 的共享网络对照：

- A2C/PPO：共享主干 → 一个 `logits` 头（离散打分）；
- SAC：**actor 是独立网络**（critic 是双 Q 且输入含动作，不适合共享）：
  主干 256-256 ReLU → 拆成两个头：
  - `mean_head` → $\mu$（每个动作维度一个均值）
  - `log_std_head` → $\log\sigma$（**对数标准差**）

为什么输出 $\log\sigma$ 而不是 $\sigma$：

- $\sigma$ 必须 $>0$，直接输出要做正约束（比如 softplus），绕；
- 神经网络线性层的输出天然无约束，$\log\sigma$ 可正可负、用的时候 $\sigma=e^{\log\sigma}$ 天然为正——**和 α 用 log_alpha 参数化是同一个套路**（4.3 节还会见到）。

代码里还有一处限幅：$\log\sigma\in[-20,\,2]$（`LOG_STD_MIN/MAX`）：

- 下限防 $\sigma$ 坍缩到 0（分布退化成确定性、$\log\pi$ 数值爆炸）；
- 上限防 $\sigma$ 过大（动作接近纯随机、训练不稳）。

### 3.2 怎么产生动作：从高斯采样，再 tanh 压缩

每次选动作分三步（`ActorNet.sample` + 训练循环）：

1. 采样 $u\sim\mathcal{N}(\mu,\sigma^2)$（各动作维度独立采样）；
2. 压缩 $a=\tanh(u)\in(-1,1)$；
3. 送环境前乘动作上限：`env.step(a * 2)`（Pendulum 真实扭矩 ∈ [-2,2]）。

两个"为什么"：

- **为什么 tanh**：高斯采样出的 $u$ 取遍 $(-\infty,+\infty)$，而真实动作是有界区间；tanh 是单调、光滑、可导的"压缩器"，把无界压到 $(-1,1)$；
- **为什么全代码统一用归一化动作**：经验池、Q 网络、$\log\pi$ 全部在 $[-1,1]$ 的**归一化动作空间**里活动，只在 `env.step` 那一行做 ×2 转换——这样 $\log\pi$ 恰好是 SAC 论文里"动作域 $[-1,1]$"上定义的概率密度，公式可以原样对应。
- 为了能反向传播，SAC 不直接采样 a，而是先从标准正态分布中采样一个独立的噪声 $\xi\sim\mathcal{N}(0,1)$，再用 $u=\mu+\sigma\xi$ 生成动作。
评估/测试时的**确定性动作**：$a=\tanh(\mu)$（不采样）——与 A2C 测试用 `argmax` 一个道理：去掉随机性、表现更稳定。

### 3.3 重参数化技巧：让"采样"这一步可导

问题：策略损失要对"从分布里采出来的动作"求梯度——但"采样"本身是随机操作，$\nabla_\theta a$ 直接写不出来。两条路线二选一：

| 路线 | 代表 | 做法 | 特点 |
|---|---|---|---|
| 似然比（log 技巧） | REINFORCE / PPO | 用 $\log\pi(a \mid s)\cdot R$ 的"事后加权"绕开采样 | 方差大、用标量权重 |
| **路径导数（重参数化）** | **SAC** | 把随机性挪到不依赖参数的噪声上 | 梯度直传、方差小 |

A2C的方法体现在代码里的 `dist.sample()` 。这里发生了采样 dist.sample()。注意：actions 是一个不可导的张量。你绝对不能用 backward() 去求 actions 对 logits 的梯度。
A2C在计算损失阶段：update 方法是`actor_loss = -(log_probs * advantages.reshape(-1)).mean()`即用 log π(a|s) · R 的事后加权绕开采样

为什么它叫“事后加权”？ 
因为你在采样的时候，根本不知道这个动作好不好。等采完动作，环境给 reward，Critic 算出 advantage 后，你回头给这个动作的概率加个权重（如果 advantage 为正，就提高这个动作的概率；为负，就降低）。这就是“事后”加权。

重参数化的具体写法（这就是 `rsample()` 干的事）：

$$u=\mu(s)+\sigma(s)\cdot\xi,\qquad \xi\sim\mathcal{N}(0,1),\qquad a=\tanh(u)$$

关键：随机性全部来自 $\xi$（与参数 $\theta$ 无关），而 $a$ 变成了 $\mu,\sigma$ 的**确定性函数**——于是可以沿着

$$a\;\leftarrow\;u\;\leftarrow\;(\mu,\sigma)\;\leftarrow\;\theta$$

一路把梯度回传（$\partial a/\partial\mu$、$\partial a/\partial\sigma$ 都可算）。torch 里 `dist.rsample()` 是可导版本，`dist.sample()` 不可导——**SAC 必须用 rsample**。

> 呼应在 Noisy Net 里见过的公式 $W=\mu^W+\sigma^W\odot\varepsilon^W$：一模一样的套路——"均值 + 尺度 × 噪声"，参数部分保持可导。当时是让"权重的噪声"可学，这里是让"动作的采样"可导。

### 3.4 log π 的修正：tanh 会"改变密度"（最容易忽略的公式）

我们要算的 $\log\pi(a|s)$ 是"**动作 $a$ 上**的对数密度"；但网络 `log_prob(u)` 给出的是"**$u$ 上**的对数密度"。$a=\tanh(u)$ 是换元，密度要被雅可比"修正"：

$$p_\pi(a|s)=\frac{p(u)}{|\mathrm{d}a/\mathrm{d}u|}=\frac{p(u)}{1-\tanh^2(u)}$$

取对数、多维动作逐维求和：

$$\log\pi(a|s)=\sum_i\Big[\log\mathcal{N}\big(u_i;\mu_i,\sigma_i^2\big)-\log\big(1-\tanh^2(u_i)\big)\Big]$$

代码（`ActorNet.sample`）：

```python
log_prob = dist.log_prob(u) - torch.log(1 - action.pow(2) + 1e-6)
log_prob = log_prob.sum(dim=-1, keepdim=True)   # 多维动作 → [batch, 1]
```

- `+1e-6`：当 $u$ 很大（tanh 饱和）时 $1-\tanh^2(u)\to 0$，$\log 0=-\infty$，加个小量兜住；
- 为什么必须修正：SAC 里 $\log\pi$ **无处不在**（Q 目标、策略损失、α 损失），如果用未修正的密度，熵项就全错了。

### 3.5 小结：一个动作的完整旅程

$$s\;\to\;\text{主干}\;\to\;(\mu,\log\sigma)\;\to\;\text{clamp}\;\to\;u=\mu+\sigma\xi\;\to\;a=\tanh(u)\;\to\;\big(a\times 2\ \text{进环境}\big)$$

$\log\pi$ 在同一条链上算出，供后续三个损失使用。


## 四、三个损失函数（全文核心）

### 4.0 总览：一次 `update()` 里谁在学什么

| 角色 | 学什么 | 损失 | 用到的数据 |
|---|---|---|---|
| actor $\pi_\theta$ | 最大化 $Q-\alpha\log\pi$ | $J_\pi$ | 池中状态 $s$ + 重参数化采样 |
| critic $Q_1,Q_2$ | 拟合软贝尔曼目标 $y$ | $\mathcal{L}_Q$ | 池中 $(s,a,r,s',d)$ |
| 温度 $\alpha$ | 让策略随机程度逼目标 | $J(\alpha)$ | 池中状态 $s$ |

更新顺序固定为：**① Q → ② π → ③ α → ④ 软更新目标网络**（对应代码里的四段，见第六章）。

### 4.1 软 Q 损失：critic 学"软贝尔曼方程"

目标（2.4 节的 $y$）：

$$y=r+\gamma(1-d)\Big[\min\big(Q_1',Q_2'\big)(s',a')-\alpha\log\pi(a'|s')\Big],\qquad a'\sim\pi_\theta$$

损失：

$$\mathcal{L}_Q=\mathrm{MSE}\big(Q_1(s,a),\,y\big)+\mathrm{MSE}\big(Q_2(s,a),\,y\big)$$

- 结构与 DQN **一模一样**："预测值 vs 停止梯度的目标"，`with torch.no_grad()` 包住 $y$ 的全部计算——DQN 笔记原理二（移动靶、为什么冻结目标）整章适用；
- 差异只有三处：单 Q → 双 Q；$\max$ → $\min$；括号里多减熵项；
- 为什么 $a'$ 用**当前策略**采样：目标定义的是"当前策略的软 Q 值"，$V$ 里的期望也是对当前策略取的——用一次采样近似期望即可（SAC 论文与主流实现的做法）。

### 4.2 策略损失：actor 学"最大化 Q 与熵的加权和"

对 actor 最小化的损失：

$$J_\pi(\theta)=\mathbb{E}_{s\sim\mathcal{D},\,\xi}\Big[\alpha\log\pi_\theta(a|s)-Q(s,a)\Big],\qquad a=\tanh\big(\mu_\theta(s)+\sigma_\theta(s)\odot\xi\big)$$

（实现里 $Q$ 取 $\min(Q_1,Q_2)$；等价写法是"最大化 $\mathbb{E}[Q-\alpha\log\pi]$"，取负号只是为了配合优化器的最小化习惯——和 REINFORCE 的负号同理。）

逐项直觉：

- $-Q(s,a)$ 项：把动作往"价值高"的方向拽；
- $+\alpha\log\pi$ 项：把**对数概率压小** = 密度更平 = 更随机（$H=\mathbb{E}[-\log\pi]$，压小 $\log\pi$ 就是抬高熵）；
- 两股力互相拉扯：Q 项想让分布"集中"到高价值动作，熵项不让它塌缩，拔河的松紧由 $\alpha$ 决定（4.3 节它自己会调）。

**与 REINFORCE 的本质区别**（重要）：

| | REINFORCE / PPO | SAC |
|---|---|---|
| 学习信号 | $\log\pi\cdot R$：用**标量回报**事后加权，好动作"概率变大"，但不知道"往哪改更好" | $-\partial Q/\partial a$：Q 网络是个光滑评价器，能直接告诉策略"动作往哪个方向挪" |
| 梯度路径 | 似然比（log 技巧） | 重参数化路径导数（3.3 节） |
| 方差 | 大 | 小（这也是 SAC 样本效率高的原因之一） |

三处技术点（对应代码）：

1. **重参数化**：$a$ 是 $\mu,\sigma$ 的确定性函数，$-\nabla_\theta Q$ 可以经 $\dfrac{\partial Q}{\partial a}\cdot\dfrac{\partial a}{\partial\theta}$ 直传；
2. **$\min(Q_1,Q_2)$**：用保守估计指导策略改进（与 4.1 的口径一致）；
3. **`self.alpha.detach()`**：$\alpha$ 在策略损失里**视为常数**（它有自己的优化器），detach 掉防止梯度"串门"到 `log_alpha` 参数上。

### 4.3 自动温度 α（SAC v2 的关键改进）

**固定 α 的痛点**：α 太大 → 学得慢（太随机）；太小 → 探索不足（早熟收敛）；不同任务、不同训练阶段最优的 α 还不一样，手调成本高。SAC v2 让 **α 自己学**。

对 $\alpha$ 的损失（最小化）：

$$J(\alpha)=\mathbb{E}_{a\sim\pi}\Big[-\alpha\big(\log\pi(a|s)+\bar{\mathcal{H}}\big)\Big],\qquad \bar{\mathcal{H}}=-\dim(\mathcal{A})$$

代码里 $\bar{\mathcal{H}}$ 就叫 `target_entropy`（$=-\text{action\_dim}$，Pendulum 为 $-1$）。论文把它叫"目标熵"，不用抠字面——**把它理解为 α 的"目标线"**：让策略的平均随机程度停在此处。

推导更新方向（两行）：

1. 对 $\alpha$ 求偏导（$\log\pi$ 视为常数）：$\dfrac{\partial J}{\partial\alpha}=-\mathbb{E}\big[\log\pi+\bar{\mathcal{H}}\big]$；
2. 但代码学的是 $\log\alpha$（保证 $\alpha>0$ 的参数化，套路同 $\log\sigma$）。用链式法则 $\dfrac{\partial J}{\partial\log\alpha}=\alpha\cdot\dfrac{\partial J}{\partial\alpha}=-\alpha\,\mathbb{E}\big[\log\pi+\bar{\mathcal{H}}\big]$。

而代码里的损失

```python
alpha_loss = -(self.log_alpha.exp() * (log_pi + self.target_entropy).detach()).mean()
```

对 `log_alpha` 求导，结果恰好等于上面第 2 条——**代码与公式完全对应**。（`.detach()` 保证梯度只流向 `log_alpha`，不许"借道"策略网络。）

由梯度下降方向，α 的行为如下：

| 情形 | 判据 | α 的调整 | 效果 |
|---|---|---|---|
| 策略比目标**更随机** | $\mathbb{E}[\log\pi]+\bar{\mathcal{H}}<0$ | α ↓ | 降低熵的权重，让 Q 项主导（转向利用） |
| 策略比目标**更确定** | $\mathbb{E}[\log\pi]+\bar{\mathcal{H}}>0$ | α ↑ | 提高熵的权重，逼策略保持探索 |

平衡点：$\mathbb{E}[\log\pi]\approx-\bar{\mathcal{H}}$。α 自己会滑到这个点上——**探索强度不用手调**。

> 不用背公式的记忆版：α 是"探索旋钮"。策略太放飞 → 拧紧（α↓）；策略太保守 → 放松（α↑）。
> 实测：本次训练 α 从 0.44 一路降到 0.14，正是"前期多探索、后期收敛"的正常形态（第七章有完整日志）。

### 4.4 四个更新为什么互不干扰（读代码易晕点）

一次 `update()` 里梯度会"流经"别的网络，但只有该更新的参数会动：

| 更新 | 梯度流经 | 怎么防止污染 |
|---|---|---|
| ① Q | $y$ 的计算全部 `no_grad` | 目标里的 $Q'$ 视为常数（DQN 同款） |
| ② actor | 要经 $Q$ 求 $\partial Q/\partial a$——**必须保留** $Q$ 的图 | `actor_optimizer` 只持有 actor 参数；流到 $Q$ 参数上的残余梯度会在下一轮 `q_optimizer.zero_grad()` 被清掉 |
| ③ α | `log_pi`、`target_entropy` 都 `.detach()` | `alpha_optimizer` 只持有 `log_alpha` |
| ④ 软更新 | `with torch.no_grad()` + 直接改 `.data` | 目标网络不参与任何反向传播 |

一句话：**"谁被更新"由优化器持有的参数决定，"谁能传梯度"由 `no_grad / detach / 保留计算图` 切片决定**——逐行读代码时按这张表核对即可。


## 五、稳定化设计

### 5.1 双 Q 取 min：DDQN 的"连续版兄弟"

- 回忆 DDQN（DQN 笔记第三章）：单 Q + "自己选自己评" → $\max$ 会系统性高估；DDQN 把"选动作"和"评价值"分给两个网络，消掉了"选中的正噪声"；
- SAC 的敌人相同（**自举高估**），但战场不同：连续动作没有离散的 $\max$ 枚举，误差来源变成"网络外推 + 多轮自举相互放大"——训练中 Q 值会整体虚高；
- 武器：**训练两个独立 Q，目标里取 $\min(Q_1',Q_2')$**（TD3 提出的思路）：
  - 两个估计各带噪声，取**小者** → 结果天然偏保守 → 与自举的虚高相互抵消，回到真值附近；
  - 代价：多一个网络，计算量增加；
- 策略改进时同样用 $\min(Q_1,Q_2)$（4.2 节），保证"学的方向也是保守的"。

> 对比 DDQN 一句话：DDQN 是"**换人评分**"（选评解耦），SAC 是"**两个人打分、取低分**"（双 Q 取 min）——都在对付同一个敌人。

### 5.2 目标网络软更新：靶子缓缓漂移

$$\theta'\;\leftarrow\;\tau\,\theta+(1-\tau)\,\theta',\qquad \tau=0.005\ \text{（每步执行）}$$

每步把目标网络往当前网络"挪 0.5%"，数学上叫 **Polyak 平均**（参数的指数滑动平均）。

- 对比 DQN 的硬拷贝（每 10 回合"瞬移"一次）：软更新的目标值是**连续缓动**的，回归目标不会跳变；
- 呼应 DQN 笔记的"移动靶"比喻：硬拷贝是"每 10 步把靶子瞬移一次"；软更新是"靶子缓缓漂移，你一直追得上"；
- 为什么 SAC 要换软更新：它**每步都学习**，目标也应该每步轻微跟随——节奏匹配才稳。

### 5.3 数值稳定性清单（代码里那些不起眼的小东西）

| 代码 | 位置 | 作用 |
|---|---|---|
| `log_std` clamp 到 [-20, 2] | `ActorNet.forward` | 防 $\sigma$ 坍缩/爆炸 |
| `+ 1e-6` | `torch.log(1 - action.pow(2) + 1e-6)` | 防 tanh 饱和时 $\log 0=-\infty$ |
| `exp(log_alpha)` | α 参数化 | 保证 $\alpha>0$ |
| `(1 - dones)` | Q 目标 | 终止状态不 bootstrap |
| `torch.no_grad()` 包住 $y$ | Q 目标 | 目标停梯度（DQN 同款） |
| 各处 `.detach()` | actor / α 损失 | 防梯度串扰（4.4 节表） |


## 六、代码 ↔ 公式对照（`SAC_train.py` 的 `update()`）

### 6.1 四个子步骤

**① 更新双 Q（4.1 节）**

```python
with torch.no_grad():
    next_actions, next_log_probs = self.actor.sample(next_states)
    q1_next = self.q1_target(next_states, next_actions)
    q2_next = self.q2_target(next_states, next_actions)
    q_next  = torch.min(q1_next, q2_next) - self.alpha * next_log_probs
    backup  = rewards + GAMMA * (1.0 - dones) * q_next

q1 = self.q1(states, actions)
q2 = self.q2(states, actions)
q_loss = F.mse_loss(q1, backup) + F.mse_loss(q2, backup)
```

↔ 目标 $y=r+\gamma(1-d)\left[\min(Q_1',Q_2')-\alpha\log\pi(a'|s')\right]$，损失 $\mathcal{L}_Q=\mathrm{MSE}(Q_1,y)+\mathrm{MSE}(Q_2,y)$。

**② 更新 actor（4.2 节）**

```python
pi, log_pi = self.actor.sample(states)
q1_pi = self.q1(states, pi)
q2_pi = self.q2(states, pi)
q_pi_min = torch.min(q1_pi, q2_pi)
actor_loss = (self.alpha.detach() * log_pi - q_pi_min).mean()
```

↔ $J_\pi=\mathbb{E}\left[\alpha\log\pi-Q\right]$（$Q=\min(Q_1,Q_2)$，$\alpha$ 视作常数）。

**③ 更新温度 α（4.3 节）**

```python
alpha_loss = -(self.log_alpha.exp() * (log_pi + self.target_entropy).detach()).mean()
```

↔ $J(\alpha)=\mathbb{E}\left[-\alpha(\log\pi+\bar{\mathcal{H}})\right]$，$\bar{\mathcal{H}}=$ `target_entropy` $=-\text{action\_dim}$。

**④ 软更新目标网络（5.2 节）**

```python
target_param.data.mul_(1 - TAU)
target_param.data.add_(TAU * param.data)
```

↔ $\theta'\leftarrow\tau\theta+(1-\tau)\theta'$。

### 6.2 训练循环里的三个约定

| 代码 | 约定 |
|---|---|
| `env.step(action * ACTION_SCALE)` | 归一化动作 $[-1,1]$ × 2 → 真实扭矩 $[-2,2]$；经验池存**归一化动作** |
| 前 `START_STEPS` 步随机 → 之后策略采样 | 预热期对应 DQN 的"ε=1 阶段"，把池子先填满多样数据 |
| 评估 `select_action(state, eval_mode=True)` | 确定性动作 $a=\tanh(\mu)$，去掉采样噪声 |


## 七、实测观察：训练日志怎么读

本次实验记录（`EPISODES=100`，共 2 万步）：

```text
回合   20 | 步数   4000 | 最近20回合平均回报 -1171.39 | α = 0.441
回合   60 | 步数  12000 | 最近20回合平均回报  -737.62 | α = 0.386
回合   80 | 步数  16000 | 最近20回合平均回报  -151.63 | α = 0.248
回合  100 | 步数  20000 | 最近20回合平均回报  -124.78 | α = 0.143
           [评估] 确定性策略 5 局平均回报:  -193.03
测试 1: -236.35 ｜ 测试 2: -256.58 ｜ 测试 3: -115.48
```

| 现象 | 解读 |
|---|---|
| 回合 20 回报 -1171 | 前 1 万步是**随机预热**，-1171 ≈ 乱转水平，正常 |
| 回合 80–100 训练回报 -152 → -124 | 学习曲线陡峭上升，策略在快速进步 |
| α：0.44 → 0.14 | 自动温度从"探索档"滑向"利用档"，**这正是 4.3 节行为表的实际演示** |
| 评估 -193 / 测试均值 -203 | 两个口径一致 → 当前真实水平约 **-200**（较优但未到 -150 档） |
| 测试好局 -115 | 随机初始角度有利时，策略能稳稳立住（甚至超过参考线 -150） |
| 测试差局 -257 | 大角度起步时还稳不住 → **训练量不足**的典型表现 |

结论：**实现正确、超参合理，只差训练量**。SAC 在 Pendulum 上通常需 6 万~20 万步才稳定到 -150 附近，继续提升的选项：

1. `EPISODES` 调到 300~1000（6 万~20 万步）重训，预期评估 -150 附近；
2. 可选：`GAMMA` 0.99 → 0.98（Pendulum 回合只有 200 步，短视野更合适）；
3. 测试多跑几局（如 10 局）再下结论，单局波动大。


## 八、与已学算法的总对比 + 一句话总结

| 维度 | DQN | A2C / PPO | SAC |
|---|---|---|---|
| 直接学什么 | $Q(s,a)$ | 离散策略 $\pi$（Categorical） | **连续高斯策略** + 软双 Q |
| 数据来源 | off-policy 回放池 | on-policy（用完即弃） | off-policy 回放池 |
| 目标网络 | 硬拷贝（每 10 回合） | 无（靠 GAE bootstrap） | **软更新**（每步，$\tau=0.005$） |
| 高估对策 | DDQN：选评解耦 | — | **双 Q 取 min** |
| 探索机制 | ε-贪婪（手调衰减） | 熵正则（辅助项） | **最大熵 + 自动温度 α** |
| 更新节奏 | 每步 | 攒批 × 多 epoch | 每步 |
| 动作空间 | 离散 | 离散（本实现） | **连续** |
| 核心目标式 | $y=r+\gamma\max Q'$ | clip 比值目标 | $y=r+\gamma\left[\min Q'-\alpha\log\pi\right]$ |

### 一句话总结

**SAC = 软贝尔曼方程（把 $-\alpha\log\pi$ 塞进自举目标）+ 可导高斯策略（重参数化 + tanh 修正）+ 回放池（off-policy）+ 自动温度（α 自己找平衡）+ 双 Q/软更新（保稳定）。**

从"零件"角度看：SAC 没有发明新零件，只有**一处焊接方式的改变**——"熵"从损失里的配角（A2C/PPO）变成目标的组成部分；其余全是 DQN 和策略梯度笔记里的老朋友。能用"已学零件清单 + 一处改动"把这篇文章复述出来，SAC 就算学明白了。
